<a href="https://colab.research.google.com/github/rah-ul6958/Day9_Be-Practical/blob/main/Day9_Lab_Building_with_LLM_APIs_Groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CELL 0: ENVIRONMENT SETUP & SECURE API AUTHENTICATION
# Run this cell first. Installs the Groq SDK, Pydantic, and dotenv.
# ==============================================================================
!pip install -q -U groq pydantic python-dotenv tabulate

import os
import sys
import time
import json
import random
import getpass
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Official Groq SDK (OpenAI-compatible client shape)
import groq
from groq import Groq

# ------------------------------------------------------------------------------
# Secure Groq API Key Ingestion (Zero-Hardcoding Policy)
# ------------------------------------------------------------------------------
# Never hardcode API keys directly in scripts!
# In Google Colab, use the Secrets Manager (icon on the left panel) as 'GROQ_API_KEY'.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass.getpass("Enter your Groq API Key: ")
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# Initialize Client
client = Groq(api_key=GROQ_API_KEY)
print("Groq API Client initialized successfully!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 8.8 MB/s eta 0:00:00
Groq API Client initialized successfully!


In [2]:
# ==============================================================================
# SECTION 1: API ARCHITECTURE, STATELESSNESS & SECURITY HYGIENE
# ==============================================================================
"""
1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:
   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.
   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.
   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.

2. THE STATELESSNESS MENTAL MODEL:
   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.
   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list
     of previous (user, assistant) turns and pass the cumulative array on every subsequent call.

3. MESSAGE ROLES MAPPING:
   Groq's Chat Completions API is OpenAI-compatible, so its roles map directly:
   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐
   │ Role Type            │ OpenAI / Anthropic      │ Groq API                  │
   ├──────────────────────┼─────────────────────────┼───────────────────────────┤
   │ System Persona/Rules │ role: 'system'          │ role: 'system'            │
   │ User Message         │ role: 'user'            │ role: 'user'              │
   │ Model Response       │ role: 'assistant'       │ role: 'assistant'         │
   └──────────────────────┴─────────────────────────┴───────────────────────────┘
"""


"\n1. WHY API KEYS MUST NEVER BE COMMITTED TO GITHUB:\n   - Automated scrapers search public GitHub commits 24/7 for exposed API keys.\n   - Leaked keys result in quota exhaustion, unexpected billing charges, and credential revocation.\n   - DEFENSE: Store keys in a local `.env` file and add `.env` to `.gitignore`.\n\n2. THE STATELESSNESS MENTAL MODEL:\n   - LLMs are 100% STATELESS: The model remembers NOTHING between individual API calls.\n   - To build a conversational multi-turn chatbot, YOU (the developer) must maintain a history list\n     of previous (user, assistant) turns and pass the cumulative array on every subsequent call.\n\n3. MESSAGE ROLES MAPPING:\n   Groq's Chat Completions API is OpenAI-compatible, so its roles map directly:\n   ┌──────────────────────┬─────────────────────────┬───────────────────────────┐\n   │ Role Type            │ OpenAI / Anthropic      │ Groq API                  │\n   ├──────────────────────┼─────────────────────────┼───────────────────────────

In [3]:
# ==============================================================================
# SECTION 2: PRE-FLIGHT TOKEN COUNTING & FINANCIAL COST ESTIMATION (UPDATED)
# ==============================================================================
"""
COST ESTIMATION BEST PRACTICE:
Count input tokens BEFORE invoking expensive generation calls to protect budget thresholds.

NOTE ON GROQ TOKEN COUNTING:
Unlike Gemini, the Groq API does not expose a dedicated "count_tokens" endpoint.
Instead, we approximate the token count locally using tiktoken's cl100k_base
encoding (the same one used by GPT models). This is an approximation because
Groq's Llama-family models use a slightly different tokenizer internally, but
it is close enough for a pre-flight cost estimate.
"""
import numpy as np
import tiktoken

# Reuse a single tokenizer instance for all estimates in this notebook
_approx_tokenizer = tiktoken.get_encoding("cl100k_base")

def preflight_cost_estimate(
    text_prompt: str,
    model_name: str = "openai/gpt-oss-120b",
    expected_output_tokens: int = 500
) -> Dict[str, Any]:
    """Estimates input tokens (approximate) and financial cost before calling the API."""
    # Approximate token count locally, since Groq has no official tokenizer endpoint
    input_tokens = len(_approx_tokenizer.encode(text_prompt))

    # Official Groq Rates per 1M tokens (USD), current as of the Groq pricing page
    pricing = {
        "openai/gpt-oss-120b": {"in": 0.15, "out": 0.60},
        "openai/gpt-oss-20b":      {"in": 0.075, "out": 0.30},
    }
    rate = pricing.get(model_name, pricing["openai/gpt-oss-120b"])

    est_cost = (input_tokens / 1e6 * rate["in"]) + (expected_output_tokens / 1e6 * rate["out"])

    return {
        "model": model_name,
        "input_tokens_approx": input_tokens,
        "estimated_output_tokens": expected_output_tokens,
        "estimated_cost_usd": np.round(est_cost, 6),
        "cost_per_10k_calls": np.round(est_cost * 10000, 2)
    }

sample_payload = "Please summarize the last 10 quarterly financial filings of Apple, Microsoft, and Google."
estimate = preflight_cost_estimate(sample_payload, model_name="openai/gpt-oss-120b")
print("=== PRE-FLIGHT TOKEN & COST AUDIT ===")
for k, v in estimate.items():
    print(f"- {k:25s}: {v}")


=== PRE-FLIGHT TOKEN & COST AUDIT ===
- model                    : openai/gpt-oss-120b
- input_tokens_approx      : 17
- estimated_output_tokens  : 500
- estimated_cost_usd       : 0.000303
- cost_per_10k_calls       : 3.03


In [4]:
# ==============================================================================
# SECTION 3: PRODUCTION RESILIENCE — EXPONENTIAL BACKOFF & RETRY LOOP
# ==============================================================================
"""
HANDLING API FAILURES IN PRODUCTION:
1. Rate Limits (HTTP 429): Hit requests-per-minute (RPM) or tokens-per-minute (TPM) ceiling.
2. Transient Server Errors (HTTP 500 / 503): Temporary Groq infrastructure hiccup.
3. Network Timeouts: Connection dropped mid-request.

REMEDY: EXPONENTIAL BACKOFF WITH JITTER:
Wait time = (base_delay * 2^attempt) + random_jitter
Prevents "Thundering Herd" problem where all failed clients retry at the exact same millisecond.
"""

def execute_with_exponential_backoff(
    api_call_func,
    max_retries: int = 4,
    base_delay: float = 1.5
):
    """Wraps an API call in an exponential backoff retry loop with random jitter."""
    for attempt in range(max_retries):
        try:
            return api_call_func()
        except (groq.APIStatusError, groq.APIConnectionError) as e:
            if attempt == max_retries - 1:
                print(f"Max retries reached. Fatal API Error: {e}")
                raise e
            # Calculate backoff delay with jitter
            delay = (base_delay * (2 ** attempt)) + random.uniform(0.1, 0.8)
            status_code = getattr(e, "status_code", "N/A")
            print(f"Warning: Transient API Error ({status_code}). Retrying in {delay:.2f}s... (Attempt {attempt+1}/{max_retries})")
            time.sleep(delay)


In [5]:
# ==============================================================================
# SECTION 4: REUSABLE GROQ WRAPPER & 3-TURN CHAT
# ==============================================================================
# Construct a reusable production function supporting:
# - Streaming responses (Low Time-To-First-Token, Groq's specialty)
# - System instructions
# - Dynamic temperature
# - Exponential backoff retry logic

def groq_call(
    prompt: str,
    system_instruction: str = "You are a concise, helpful enterprise AI assistant.",
    temperature: float = 0.2,
    stream: bool = False,
    model: str = "openai/gpt-oss-120b"
) -> str:
    """Production-grade wrapper for the Groq API with error handling and streaming."""
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": prompt}
    ]

    if stream:
        def stream_call():
            full_text = []
            response_stream = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=800,
                stream=True
            )
            for chunk in response_stream:
                delta = chunk.choices[0].delta.content
                if delta:
                    print(delta, end="", flush=True)
                    full_text.append(delta)
            print() # Print final newline
            return "".join(full_text)

        return execute_with_exponential_backoff(stream_call)
    else:
        def standard_call():
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=800
            )
            return resp.choices[0].message.content.strip()

        return execute_with_exponential_backoff(standard_call)

# ------------------------------------------------------------------------------
# 3-Turn Conversational Memory Loop Demonstration
# ------------------------------------------------------------------------------
print("=== MULTI-TURN CONVERSATION LOOP ===")

# Explicitly maintain stateless conversation history (Groq uses OpenAI-style
# {"role": ..., "content": ...} messages, unlike Gemini's "parts" format)
conversation_history = []
system_persona = "You are a Senior PostgreSQL Database Administrator. Answer concisely in 2 sentences."

def send_chat_turn(user_message: str):
    print(f"\nUser: {user_message}")
    print("Assistant: ", end="")

    # 1. Append user message to history
    conversation_history.append({"role": "user", "content": user_message})

    # 2. Call Groq passing the system persona plus the full conversation history
    messages = [{"role": "system", "content": system_persona}] + conversation_history
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=0.0
    )

    bot_reply = response.choices[0].message.content.strip()
    print(bot_reply)

    # 3. Append model response to history to maintain context
    conversation_history.append({"role": "assistant", "content": bot_reply})

# Execute 3-Turn Dialogue (Demonstrating Context Memory)
send_chat_turn("What is the difference between a clustered and non-clustered index?")
send_chat_turn("Which one is faster for range queries on primary keys?") # Pronoun resolution!
send_chat_turn("Can a table have multiple of the faster one?")           # Contextual follow-up!


=== MULTI-TURN CONVERSATION LOOP ===

User: What is the difference between a clustered and non-clustered index?
Assistant: A clustered index (implemented via `CLUSTER` in PostgreSQL) reorders the table’s heap so that the rows are stored physically in the same order as the index keys, and only one such index can be active per table. A non‑clustered index leaves the heap unchanged and maintains a separate B‑tree that points to the row locations, allowing many indexes per table.

User: Which one is faster for range queries on primary keys?
Assistant: A clustered index is generally faster for range queries on primary‑key values because the rows are stored sequentially on disk, so a scan can read them in order with minimal random I/O. If the table isn’t clustered, the same range scan must repeatedly jump from the index to disparate heap pages, which is slower.

User: Can a table have multiple of the faster one?
Assistant: No; a PostgreSQL table can be physically ordered (clustered) by only 

In [6]:
# ==============================================================================
# SECTION 5: STUDENT LAB WORKSPACE (PORTFOLIO APPLICATION)
# ==============================================================================
"""
STUDENT LAB ASSIGNMENT:
Build an end-to-end AI Application: "The Executive Resume Bullet & Impact Optimizer"

APPLICATION REQUIREMENTS:
1. Structured JSON Schema (Pydantic):
   - `original_bullet`: Raw user text
   - `xyz_formatted_bullet`: Rewritten using Google's XYZ Formula:
     "Accomplished [X], as measured by [Y], by doing [Z]"
   - `impact_metric`: The quantifiable numeric KPI
   - `action_verb`: Strong opening action verb
   - `seniority_score`: Integer rating (1 to 10) of executive presence
   - `critique`: 1-sentence explanation of what was improved
2. Interactive Revision History: Allow user to request a revision (multi-turn).
3. Streaming or Schema Parsing: Correctly parse and display output.
4. Error Handling: Enclose calls in retry blocks.
"""

# ==============================================================================
# TASK 1: DEFINE PYDANTIC SCHEMA FOR STRUCTURED RESUME OPTIMIZATION
# ==============================================================================

# TODO 1.1: Complete the Pydantic Schema
class ResumeBulletOptimization(BaseModel):
    """Structured output schema for one optimized resume bullet point."""
    original_bullet: str = Field(description="The raw resume bullet text typed by the user")
    xyz_formatted_bullet: str = Field(description="The rewritten bullet using Google's XYZ Formula: 'Accomplished [X], as measured by [Y], by doing [Z]'")
    impact_metric: str = Field(description="The quantifiable numeric KPI extracted or inferred from the bullet, e.g. '35% cost reduction'")
    action_verb: str = Field(description="A single strong opening action verb used in the rewritten bullet")
    seniority_score: int = Field(ge=1, le=10, description="Integer rating from 1 (junior/weak) to 10 (highly executive) of how senior the bullet sounds")
    critique: str = Field(description="One sentence explaining what was improved and why")

# ==============================================================================
# TASK 2: BUILD THE APPLICATION ENGINE
# ==============================================================================

# Rate-limit pacing: the Groq free tier allows 30 requests/minute on
# openai/gpt-oss-120b, so a small 2.5s gap between calls is more than
# enough headroom (60s / 30 = 2s, +0.5s of safety margin).
_last_api_call_time = 0.0
_min_seconds_between_calls = 2.5

def optimize_resume_bullet(raw_bullet: str, revision_feedback: str = None) -> ResumeBulletOptimization:
    """
    Sends a weak resume bullet to Groq and gets back a structured, XYZ-formatted rewrite.
    If revision_feedback is given, it asks Groq to revise its own previous suggestion,
    which is what powers the multi-turn "Interactive Revision History" feature.
    """

    # Build the base instruction prompt for the model
    prompt_text = f"""
You are an expert executive resume writer.

Rewrite the following weak resume bullet point using Google's XYZ Formula:
"Accomplished [X], as measured by [Y], by doing [Z]"

Weak bullet: "{raw_bullet}"

Return your answer strictly following the provided JSON schema.
"""

    # If the user asked for a revision, add their feedback to the prompt
    if revision_feedback:
        prompt_text += f"""
The user was not fully satisfied with a previous version and gave this revision request:
"{revision_feedback}"

Please revise the bullet again, taking this feedback into account.
"""

    messages = [
        {"role": "system", "content": "You are an expert executive resume writer. Always respond with valid JSON matching the given schema."},
        {"role": "user", "content": prompt_text}
    ]

    # TODO 2.1: Configure the Groq call with temperature=0.1 and a JSON Schema response_format
    # (Groq's equivalent of Gemini's response_mime_type='application/json' + response_schema)
    def api_call():
        return client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            temperature=0.1,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "resume_bullet_optimization",
                    "schema": ResumeBulletOptimization.model_json_schema()
                }
            }
        )

    # TODO 2.2: Execute API call with exponential backoff
    # Pace ourselves so we don't exceed the free-tier limit of 30 requests/minute.
    # This runs BEFORE every call, so it applies no matter how many times
    # optimize_resume_bullet() gets called across Task 3.
    global _last_api_call_time
    seconds_since_last_call = time.time() - _last_api_call_time
    if seconds_since_last_call < _min_seconds_between_calls:
        wait_time = _min_seconds_between_calls - seconds_since_last_call
        print(f"Pausing {wait_time:.1f}s to stay under the free-tier rate limit...")
        time.sleep(wait_time)

    response = execute_with_exponential_backoff(api_call)
    _last_api_call_time = time.time()

    # Groq returns the JSON as a plain string in the message content, so we
    # parse it ourselves into our Pydantic schema (there is no auto-parsing
    # .parsed attribute like the Gemini SDK provides).
    raw_json_text = response.choices[0].message.content
    result = ResumeBulletOptimization(**json.loads(raw_json_text))

    return result


def display_optimization_result(result: ResumeBulletOptimization):
    """Nicely prints one ResumeBulletOptimization result to the console."""
    print(f"Original Bullet:   {result.original_bullet}")
    print(f"XYZ Rewrite:       {result.xyz_formatted_bullet}")
    print(f"Impact Metric:     {result.impact_metric}")
    print(f"Action Verb:       {result.action_verb}")
    print(f"Seniority Score:   {result.seniority_score}/10")
    print(f"Critique:          {result.critique}")


# ==============================================================================
# TASK 3: TEST APPLICATION ON REAL-WORLD WEAK BULLETS
# ==============================================================================

weak_resume_bullets = [
    "Responsible for managing a team and improving processes.",
    "Worked on the company website and fixed some bugs.",
    "Helped increase sales for the department."
]

optimization_results = []

print("=== RUNNING RESUME BULLET OPTIMIZER ON REAL-WORLD WEAK BULLETS ===\n")

for bullet in weak_resume_bullets:
    print(f'--- Optimizing: "{bullet}" ---')
    result = optimize_resume_bullet(bullet)
    display_optimization_result(result)
    optimization_results.append(result)
    print()

# ------------------------------------------------------------------------------
# Interactive Revision History Demo (Multi-Turn)
# ------------------------------------------------------------------------------
# This demonstrates requirement #2: allowing the user to request a revision
# on a bullet that was already optimized, simulating a real back-and-forth.
print("=== INTERACTIVE REVISION HISTORY DEMO ===\n")

bullet_to_revise = weak_resume_bullets[0]

print(f'Bullet being revised: "{bullet_to_revise}"\n')
print("First version:")
first_version = optimize_resume_bullet(bullet_to_revise)
display_optimization_result(first_version)

# The user reviews the first version and is not fully happy, so they ask for a change
revision_request = "Make the seniority score higher and use a more powerful action verb."
print(f'\nUser revision request: "{revision_request}"\n')

print("Revised version:")
revised_version = optimize_resume_bullet(bullet_to_revise, revision_feedback=revision_request)
display_optimization_result(revised_version)

# Keep a simple revision history list so both versions can be compared/shown later
revision_history = [first_version, revised_version]

print(f"\nRevision history saved with {len(revision_history)} version(s).")


=== RUNNING RESUME BULLET OPTIMIZER ON REAL-WORLD WEAK BULLETS ===

--- Optimizing: "Responsible for managing a team and improving processes." ---
Original Bullet:   Responsible for managing a team and improving processes.
XYZ Rewrite:       Accomplished a 20% reduction in process cycle time, as measured by cycle time metrics, by leading a cross-functional team of 8 to implement Lean methodologies.
Impact Metric:     20% reduction in process cycle time
Action Verb:       Led
Seniority Score:   7/10
Critique:          Added a concrete action verb, quantified the impact, and specified how it was achieved, turning a vague statement into a results‑focused bullet.

--- Optimizing: "Worked on the company website and fixed some bugs." ---
Pausing 2.5s to stay under the free-tier rate limit...
Original Bullet:   Worked on the company website and fixed some bugs.
XYZ Rewrite:       Optimized the company website, as measured by a 20% reduction in page load time, by fixing critical bugs and strea

In [7]:
# ==============================================================================
# SECTION 6: GIT REPOSITORY HYGIENE — CREATING .ENV AND .GITIGNORE
# ==============================================================================
"""
INSTRUCTIONS FOR PUSHING TO GITHUB SAFELY:

1. Create a `.env` file locally:
   GROQ_API_KEY=your_actual_key_here

2. Create a `.gitignore` file in your project root containing:
   .env
   .env.local
   *.joblib
   __pycache__/
   .ipynb_checkpoints/

3. In your Python script (`app.py`), load the key cleanly via:
   from dotenv import load_dotenv
   load_dotenv()
   api_key = os.getenv("GROQ_API_KEY")
"""

# Script to generate .gitignore locally in Colab
with open(".gitignore", "w") as f:
    f.write(".env\n.env.*\n*.joblib\n__pycache__/\n.ipynb_checkpoints/\n")

print("'.gitignore' template created successfully!")


'.gitignore' template created successfully!
